In [ ]:
# Character-level GPT (nanoGPT-style) assembled end to end and trained on Shakespeare.
#   GPT = token embedding + learned positional embedding
#         + N pre-LN causal Transformer blocks + final LayerNorm + tied LM head.
# Trains on ~1MB of tinyshakespeare and generates text; target: val perplexity < 5.
#
# Self-contained and Colab-ready: pick a GPU runtime (Runtime > Change runtime type >
# GPU) and Run all. The causal attention here is the fused production path; the manual
# softmax(QKᵀ/√d_k)V version is built from scratch in Ass_4.ipynb (Note_4 §1-2, §7).
# Reference: github.com/karpathy/nanoGPT (config/train_shakespeare_char.py).

In [ ]:
from __future__ import annotations

import math
import os
import urllib.request
from contextlib import nullcontext
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"device: {device}")

In [ ]:
# Data: download ~1MB of tinyshakespeare, build a character vocabulary, and encode the
# whole corpus into one long tensor of token ids split 90/10 into train / val.

DATA_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/"
    "data/tinyshakespeare/input.txt"
)
DATA_PATH = "shakespeare.txt"

if not os.path.exists(DATA_PATH):
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]            # string -> list[int]
decode = lambda ids: "".join(itos[i] for i in ids)  # list[int] -> string

data = torch.tensor(encode(text), dtype=torch.long)
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]
print(f"corpus chars: {len(text):,}   vocab: {vocab_size}")
print(f"train tokens: {len(train_data):,}   val tokens: {len(val_data):,}")

In [ ]:
# Hyperparameters (nanoGPT char config) + a random batch sampler. Each batch grabs B
# windows of length block_size; targets y are the inputs x shifted one step right, so
# position t is trained to predict token t+1.

@dataclass
class GPTConfig:
    vocab_size: int = vocab_size
    block_size: int = 256     # context length (chars the model sees at once)
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384         # d_model; per-head width = 384/6 = 64
    dropout: float = 0.2

cfg = GPTConfig()

# Training schedule.
batch_size = 64
max_iters = 5000
eval_interval = 250
eval_iters = 200          # batches averaged per loss estimate
learning_rate = 1e-3
min_lr = 1e-4
warmup_iters = 100
weight_decay = 1e-1
grad_clip = 1.0


def get_batch(split: str):
    src = train_data if split == "train" else val_data
    ix = torch.randint(len(src) - cfg.block_size, (batch_size,))
    x = torch.stack([src[i : i + cfg.block_size] for i in ix])
    y = torch.stack([src[i + 1 : i + 1 + cfg.block_size] for i in ix])
    return x.to(device), y.to(device)

In [ ]:
# Causal multi-head self-attention. Q/K/V are produced by one fused linear, split into
# heads, then scaled_dot_product_attention applies the √d_k scaling, the causal mask
# (is_causal=True blocks future positions), and softmax in one fused, memory-efficient
# kernel. Outputs are concatenated across heads and mixed by the output projection.

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.dropout = cfg.dropout
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)   # W_Q, W_K, W_V fused
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd)        # W_O
        self.attn_drop = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        # (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, is_causal=True,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)       # concat heads
        return self.resid_drop(self.c_proj(y))

In [ ]:
# Position-wise feed-forward (4x hidden, GELU) and the pre-LN Transformer block:
#   x = x + Attention(LayerNorm(x));  x = x + FFN(LayerNorm(x)).

class MLP(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.drop(self.c_proj(self.gelu(self.c_fc(x))))


class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [ ]:
# The full GPT. Token + positional embeddings are summed, passed through N blocks and a
# final LayerNorm, then projected to vocab logits by the LM head. The LM head shares its
# weights with the token embedding (weight tying) - fewer params, better generalization.

class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)   # token embedding
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)   # positional encoding
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layer))
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)
        self.tok_emb.weight = self.lm_head.weight                 # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size, "sequence longer than block_size"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))   # (B, T, n_embd)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                # (B, T, vocab_size)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size:]   # crop to context window
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature    # last-step logits
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


model = GPT(cfg).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

In [ ]:
# Optimizer + schedule. AdamW with cosine learning-rate decay after a short linear
# warmup. estimate_loss averages the loss over many batches (no grad, eval mode) for a
# low-variance train/val read used both for logging and for the final perplexity.

optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate,
    betas=(0.9, 0.99), weight_decay=weight_decay,
)

# Mixed precision on CUDA (bf16 if supported) for speed; plain fp32 elsewhere.
use_amp = device == "cuda"
amp_dtype = torch.bfloat16 if (use_amp and torch.cuda.is_bf16_supported()) else torch.float16
ctx = torch.autocast(device_type="cuda", dtype=amp_dtype) if use_amp else nullcontext()
scaler = torch.cuda.amp.GradScaler(enabled=use_amp and amp_dtype == torch.float16)


def get_lr(it: int) -> float:
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))   # cosine 1 -> 0
    return min_lr + coeff * (learning_rate - min_lr)


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            x, y = get_batch(split)
            with ctx:
                _, loss = model(x, y)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
# Training loop. ~5000 iters; on a Colab T4/L4 GPU this is roughly 15-25 min and reaches
# val loss ~1.47 (perplexity ~4.3). Keeps the best val checkpoint in memory.

best_val = float("inf")
best_state = None
model.train()

for it in range(max_iters + 1):
    for g in optimizer.param_groups:
        g["lr"] = get_lr(it)

    if it % eval_interval == 0 or it == max_iters:
        m = estimate_loss()
        print(f"iter {it:5d} | train {m['train']:.4f} | val {m['val']:.4f} "
              f"| val ppl {math.exp(m['val']):.3f} | lr {get_lr(it):.2e}")
        if m["val"] < best_val:
            best_val = m["val"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if it == max_iters:
        break

    x, y = get_batch("train")
    with ctx:
        _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    scaler.step(optimizer)
    scaler.update()

print(f"\nbest val loss: {best_val:.4f}  ->  best val perplexity: {math.exp(best_val):.3f}")

In [ ]:
# Evaluate the best checkpoint: report final val perplexity (target < 5) and generate a
# fresh sample, primed with a newline, to eyeball coherence.

if best_state is not None:
    model.load_state_dict(best_state)

final = estimate_loss()
val_ppl = math.exp(final["val"])
print(f"final val loss {final['val']:.4f}  |  val perplexity {val_ppl:.3f}")
print(f"perplexity < 5 target: {'PASS' if val_ppl < 5 else 'FAIL'}\n")

context = torch.zeros((1, 1), dtype=torch.long, device=device)  # newline token id 0
sample = model.generate(context, max_new_tokens=500, temperature=0.8, top_k=200)
print(decode(sample[0].tolist()))